#Creating Dimension  Dim_asset

## Overview
Creates **Dim_asset** dimension with asset configuration, maintenance status, and SLA attributes.

**Source**: asset_config_data (silver) + dim_device (gold)
**Target**: telecom_catalog.gold_schema.dim_asset

## Step 1: Create Dimension Table
Define dim_asset schema with asset_key (surrogate key), device_key (FK), asset attributes, maintenance status, and SLA fields.

In [0]:
CREATE OR REPLACE TABLE telecom_catalog.gold_schema.dim_asset
(
    asset_key BIGINT GENERATED ALWAYS AS IDENTITY,

    device_key BIGINT NOT NULL,

    asset_type STRING,
    firmware_version STRING,
    vendor STRING,
    install_date DATE,

    device_age_category STRING,

    maintenance_status STRING,
    maintenance_required STRING,

    sla_level STRING,
    sla_priority INT,

    rack_location STRING,
    region STRING,
    support_team STRING
);

## Step 2: Verify Schema
Validate table structure and column definitions.

In [0]:
DESCRIBE telecom_catalog.gold_schema.dim_asset;

col_name,data_type,comment
asset_key,bigint,null
device_key,bigint,null
asset_type,string,null
firmware_version,string,null
vendor,string,null
install_date,date,null
device_age_category,string,null
maintenance_status,string,null
maintenance_required,string,null
sla_level,string,null


## Step 3: Prepare Source View
Join asset_config_data with dim_device to obtain device_key for each asset.

In [0]:
CREATE OR REPLACE TEMP VIEW vw_dim_asset_source AS

SELECT
    dd.device_key,

    ac.asset_type,
    ac.firmware_version,
    ac.vendor,
    ac.install_date,

    ac.device_age_category,

    ac.maintenance_status,
    ac.maintenance_required,

    ac.sla_level,
    ac.sla_priority,

    ac.rack_location,
    ac.region,
    ac.support_team

FROM telecom_catalog.silver_schema.asset_config_data ac
INNER JOIN telecom_catalog.gold_schema.dim_device dd
ON ac.device_id = dd.device_id;

## Step 4: Preview Source Data
Validate source view before merge.

In [0]:
SELECT *
FROM vw_dim_asset_source
LIMIT 10;

device_key,asset_type,firmware_version,vendor,install_date,device_age_category,maintenance_status,maintenance_required,sla_level,sla_priority,rack_location,region,support_team
288,Load Balancer,v1.3,F5,2022-02-11,Old,Active,No,Silver,3,RACK_0122,Bangalore,CloudOps
63,Firewall,v3.3,Palo Alto,2023-03-26,Old,Maintenance Due,Yes,Gold,2,RACK_0147,Bangalore,CloudOps
457,Server,v3.8,Dell,2022-06-14,Old,Active,No,Silver,3,RACK_0153,Mumbai,InfraWest
178,Router,v2.9,Cisco,2022-12-23,Old,Active,No,Gold,2,RACK_0160,Bangalore,CloudOps
240,Firewall,v3.6,Palo Alto,2024-02-16,Old,Active,No,Gold,2,RACK_0204,Delhi,InfraNorth
114,Load Balancer,v3.8,F5,2024-07-02,Old,Active,No,Gold,2,RACK_0207,Mumbai,InfraWest
352,Server,v2.1,Dell,2021-06-20,Old,Active,No,Gold,2,RACK_0249,Hyderabad,SecurityOps
695,Load Balancer,v3.7,F5,2023-09-01,Old,Maintenance Due,Yes,Gold,2,RACK_0335,Hyderabad,SecurityOps
188,Server,v2.0,Dell,2024-05-21,Old,Maintenance Due,Yes,Silver,3,RACK_0343,Chennai,InfraSouth
803,Switch,v2.7,Juniper,2021-09-15,Old,Active,No,Gold,2,RACK_0371,Chennai,InfraSouth


## Step 5: Merge Asset Data
MERGE operation: UPDATE existing records by device_key, INSERT new assets. SCD Type 1 pattern.

In [0]:
MERGE INTO telecom_catalog.gold_schema.dim_asset AS target

USING vw_dim_asset_source AS source

ON target.device_key = source.device_key

WHEN MATCHED THEN
UPDATE SET
    target.asset_type = source.asset_type,
    target.firmware_version = source.firmware_version,
    target.vendor = source.vendor,
    target.install_date = source.install_date,
    target.device_age_category = source.device_age_category,
    target.maintenance_status = source.maintenance_status,
    target.maintenance_required = source.maintenance_required,
    target.sla_level = source.sla_level,
    target.sla_priority = source.sla_priority,
    target.rack_location = source.rack_location,
    target.region = source.region,
    target.support_team = source.support_team

WHEN NOT MATCHED THEN
INSERT (
    device_key,
    asset_type,
    firmware_version,
    vendor,
    install_date,
    device_age_category,
    maintenance_status,
    maintenance_required,
    sla_level,
    sla_priority,
    rack_location,
    region,
    support_team
)
VALUES (
    source.device_key,
    source.asset_type,
    source.firmware_version,
    source.vendor,
    source.install_date,
    source.device_age_category,
    source.maintenance_status,
    source.maintenance_required,
    source.sla_level,
    source.sla_priority,
    source.rack_location,
    source.region,
    source.support_team
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
901,0,0,901


## Step 6: Validate Load
Verify record count after merge.

In [0]:
SELECT COUNT(*)
FROM telecom_catalog.gold_schema.dim_asset;

COUNT(*)
901


---
## Summary
**Table**: `telecom_catalog.gold_schema.dim_asset`

**Key Columns**:
* asset_key (PK, auto-generated)
* device_key (FK to dim_device)
* asset_type, firmware_version, vendor, install_date
* maintenance_status, sla_level, rack_location, region

**Load**: MERGE on device_key (SCD Type 1)